# Trabajo Práctico Integrador - Introducción al Análisis de Datos
## ***Segunda Parte***: Preparación y Transformación de Datos

**Caso:** análisis y predicción de cancelaciones en reservas hoteleras.

> La empresa busca comprender qué factores influyen en la cancelación de reservas en hoteles urbanos y resort, analizando variables como fechas de estadía, tipo de cliente, canal de reserva, historial previo y tarifas, con el fin de identificar patrones y mejorar la gestión operativa.




## Desarrollado por

- **Integrantes:** `Matias Carro` y `Hugo Catalan`
- **Comisión:** 11
- **Dataset asignado:** Dataset K
- **Entrega:** Primer Entrega - Semana 3
- **Fecha:** 

## Importación de librerías

### Librerias a utilizar:

- **pandas:** para cargar el dataset y trabajar con datos en forma de tablas (DataFrames).
- **numpy:** para realizar operaciones numéricas y manejar arreglos de forma eficiente.



In [1]:
import pandas as pd
import numpy as np

print("Versión de Pandas:", pd.__version__)
print("Versión de numpy:", np.__version__)
print("\nLibrerías cargadas correctamente.")


Versión de Pandas: 3.0.5
Versión de numpy: 2.5.2

Librerías cargadas correctamente.


## 1. Presentacion del problema

El caso de estudio se centra en el análisis de reservas hoteleras con el objetivo de comprender qué factores están asociados a la cancelación de estadías. El dataset asignado contiene información detallada de cada reserva, incluyendo tipo de hotel, fechas de llegada, duración de la estadía, composición del grupo, país de origen, canal de reserva, tipo de cliente, historial previo, tarifa promedio por noche y características operativas como depósito, agente, pedidos especiales y cambios realizados.


### Relación entre datos, información y conocimiento
En este trabajo partimos de los **datos** que son valores crudos del sistema de reservas: fechas, cantidades, categorías y códigos.  
Mediante el análisis exploratorio estos datos se convierten en **información**, como distribuciones, patrones y diferencias entre reservas canceladas y no canceladas.  
A partir de esa información generamos el **conocimiento** que nos permite entender el comportamiento de los clientes y detectar factores que podrían influir en la cancelación de una reserva.  

Esta relación es clave para el caso: los datos del hotel por sí solos no dicen nada, pero al transformarlos en información y luego interpretarlos, podemos identificar variables relevantes (como `lead_time`, `deposit_type` o `customer_type`) que ayudan a explicar por qué algunas reservas se cancelan y otras no.

### Ciclo de vida del análisis
Este trabajo se enmarca en el ciclo de vida del análisis de datos, que incluye:
1. Obtención del dataset asignado.  
2. Comprensión inicial del problema (cancelaciones hoteleras).  
3. Exploración y limpieza mínima (EDA).  
4. Transformación y preparación de variables relevantes.  
5. Interpretación y comunicación de resultados.

### Variable objetivo
La **variable objetivo** del análisis es **`is_canceled`**, que indica si la reserva fue cancelada (`1`) o no (`0`).  
Su distribución será calculada y analizada en las próximas secciones para comprender el comportamiento general del conjunto de datos y orientar las preguntas del análisis.

### Preguntas iniciales que orientan el trabajo
- ¿Qué características diferencian a las reservas canceladas de las no canceladas?  
- ¿Influyen el tipo de hotel o el canal de reserva en la cancelación?  
- ¿Las reservas con mayor anticipación (`lead_time`) presentan mayor probabilidad de cancelación?  
- ¿Los clientes con pedidos especiales o estacionamiento tienden a cancelar menos?  
- ¿Las políticas de depósito (`deposit_type`) reducen la cancelación?  
- ¿Existen segmentos de mercado con mayor riesgo de cancelación?  


---

## 2. Dataset Asignado

- **Comisión:** 11
- **Dataset asignado:** Dataset K


---

## 3. Carga del dataset

Se carga el Dataset perteneciente a la comisión 11:


In [2]:
df = pd.read_csv("hotel booking TPI grupo K.csv")

print("Dataset cargado correctamente.")


Dataset cargado correctamente.


---

## 4. Resumen del dataset

Resumen de los datos 

In [3]:
filas, columnas = df.shape
print(f"El dataset tiene {filas} filas y {columnas} columnas.")

print("Columnas del dataset:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.info())


El dataset tiene 25000 filas y 32 columnas.
Columnas del dataset:
['booking_id', 'hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'arrival_date', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']

Tipos de datos:
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   booking_id                      25000 non-null  str    
 1   hotel                          

El dataset tiene 25.000 filas y 32 columnas.  
Las variables incluyen información temporal, categórica y numérica relevante para el análisis.


### Diccionario de variables 

| Variable | Traducción | Representación | Tipo de Datos |
|----------|------------|----------------|-----------|
| booking_id | ID de reserva | Identificador único de cada reserva. | str |
| hotel | Tipo de hotel | Indica si la reserva corresponde a City Hotel (Ciudad) o Resort Hotel (Resort). | str |
| is_canceled | Cancelada | Indica si la reserva fue cancelada (1) o no (0). | int64 |
| lead_time | Anticipación | Días entre la fecha de reserva y la fecha de llegada. | int64 |
| arrival_date_year | Año de llegada | Año en el que el huésped llega al hotel. | int64 |
| arrival_date_month | Mes de llegada | Mes en el que el huésped llega al hotel. | str |
| arrival_date_week_number | Semana de llegada | Número de semana del año en la que llega el huésped. | int64 |
| arrival_date_day_of_month | Día del mes de llegada | Día del mes en el que llega el huésped. | int64 |
| arrival_date | Fecha de llegada | Fecha completa de llegada (YYYY-MM-DD). | str |
| stays_in_weekend_nights | Noches de fin de semana | Cantidad de noches en fines de semana. | int64 |
| stays_in_week_nights | Noches de semana | Cantidad de noches de lunes a jueves. | int64 |
| adults | Adultos | Número de adultos en la reserva. | int64 |
| children | Niños | Número de niños en la reserva. | float64 |
| babies | Bebés | Número de bebés en la reserva. | int64 |
| meal | Tipo de comida | Plan de comidas asociado a la reserva (BB, HB, SC, etc.). | str |
| country | País | País de origen del huésped. | str |
| market_segment | Segmento de mercado | Tipo de cliente según el canal de adquisición. | str |
| distribution_channel | Canal de distribución | Canal por el cual se realizó la reserva. | str |
| is_repeated_guest | Huésped repetido | Indica si el cliente ya se alojó anteriormente. | int64 |
| previous_cancellations | Cancelaciones previas | Cantidad de reservas previas canceladas por el cliente. | int64 |
| previous_bookings_not_canceled | Reservas previas no canceladas | Cantidad de reservas previas completadas por el cliente. | int64 |
| reserved_room_type | Habitación reservada | Tipo de habitación solicitada originalmente. | str |
| assigned_room_type | Habitación asignada | Tipo de habitación finalmente asignada. | str |
| booking_changes | Cambios en la reserva | Número de modificaciones realizadas a la reserva. | int64 |
| deposit_type | Tipo de depósito | Política de depósito aplicada. | str |
| agent | Agente | Código del agente que gestionó la reserva. | float64 |
| company | Compañía | Código de la empresa asociada a la reserva. | float64 |
| days_in_waiting_list | Días en lista de espera | Tiempo que la reserva permaneció en espera antes de confirmarse. | int64 |
| customer_type | Tipo de cliente | Clasificación del cliente (Transient, Contract, Group, etc.). | str |
| adr | Tarifa promedio diaria | Precio promedio por noche de la reserva. | float64 |
| required_car_parking_spaces | Estacionamiento requerido | Cantidad de espacios de estacionamiento solicitados. | int64 |
| total_of_special_requests | Pedidos especiales | Número de solicitudes especiales realizadas por el cliente. | int64 |




---

## 5. Diagnóstico y limpieza inicial de datos.

Se comienza la limpieza de los datos

### 0. Preparativos

Se crea una copia del dataset original para conservar un respaldo íntegro de los datos. A partir de este punto, todas las modificaciones se realizarán exclusivamente sobre la copia de trabajo.

El dataset original permanece sin alteraciones y funciona como *referencia histórica* ante cualquier revisión o necesidad de volver al estado inicial.

In [4]:
df_original = df.copy()
print("Copia del dataset creada correctamente.")

Copia del dataset creada correctamente.


***Bitácora***: Se crea el documento “Bitácora” para dejar constancia de los problemas detectados, las variables afectadas, las decisiones tomadas y la justificación correspondiente.

In [5]:
registros_bitacora = []

def registrar(problema, variable, decision, justificacion):
    registros_bitacora.append({
        "problema_detectado": problema,
        "variable_afectada": variable,
        "decision_tomada": decision,
        "justificacion": justificacion
    })

### 1. Detección de valores duplicados

Antes de comenzar la limpieza, se revisa si existen registros duplicados en el dataset, ya que podrían generar sesgos al contar dos veces la misma reserva.

En este tipo de datos, dos reservas pueden tener exactamente las mismas características sin ser un error, por lo que solo se consideran duplicados aquellos casos donde todas las columnas coinciden completamente.

Como el dataset no posee un identificador único por reserva, la detección se realiza comparando todas las variables. 

In [6]:
duplicados = df.duplicated().sum()
print(f"Cantidad de registros duplicados: {duplicados}")

Cantidad de registros duplicados: 0


La revisión arrojó **0 duplicados**, por lo que no es necesario aplicar cambios en esta etapa.

---

### Identificación de valores faltantes

Se realiza un vistazo en general de los valores faltantes del dataset completo


In [7]:
faltantes = df.isnull().sum().sort_values(ascending=False)
print("Valores faltantes por variable:\n")
print(faltantes)


Valores faltantes por variable:

company                           23574
agent                              3489
country                             101
booking_id                            0
arrival_date_year                     0
arrival_date_month                    0
is_canceled                           0
hotel                                 0
arrival_date_day_of_month             0
arrival_date                          0
stays_in_week_nights                  0
stays_in_weekend_nights               0
adults                                0
children                              0
arrival_date_week_number              0
lead_time                             0
meal                                  0
babies                                0
market_segment                        0
distribution_channel                  0
previous_bookings_not_canceled        0
reserved_room_type                    0
is_repeated_guest                     0
previous_cancellations                0
booking

**Diagnóstico inicial de valores faltantes**

***1. company - 23.574 faltantes***

Código administrativo de empresa asociada a la reserva, la mayoría de las reservas no pertenecen a ninguna compañía.  

No aporta información analítica útil para explicar cancelaciones.  

***2. agent - 3.489 faltantes***

Código interno del agente que gestionó la reserva, los faltantes representan reservas hechas sin agente.  
No es una categoría analítica, sino un identificador, no importa informacion para analizar el motivo de las cancelaciones.  

***3. country - 101 faltantes***

Única variable relevante con faltantes reales, los 101 registros sin país pueden deberse a reservas incompletas, carga manual o canales que no reportan país.  
El porcentaje es muy bajo, se debe verificar que hacer con la variable. Se vera en etapas proximas.

---

### 2. Selección de variables

Como parte del análisis inicial, se revisan todas las columnas del dataset para identificar variables que no aporten información relevante al estudio o que no resulten útiles para el análisis de cancelaciones. Este paso permite simplificar el dataset y concentrar el trabajo en las variables que realmente contribuyen a explicar el comportamiento de las reservas.

En nuestro caso, se inspeccionan nuevamente las columnas del DataFrame y se evalúa si alguna corresponde a identificadores irrelevantes, códigos internos, variables redundantes o información que no será utilizada en el análisis. Tras esta revisión, se determina si es necesario eliminar alguna columna o si todas deben conservarse para las etapas posteriores.

In [8]:
lista_variables = df.columns.tolist()
print(f"Listado de variables:\n{lista_variables}")

print("\nVista inicial de la tabla:\n")
df.head()


Listado de variables:
['booking_id', 'hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'arrival_date', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']

Vista inicial de la tabla:



,booking_id,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,arrival_date,stays_in_weekend_nights,...,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,HB-014269,Resort Hotel,0,17,2025,February,8,21,2025-02-21,0,...,E,0,No Deposit,NaN,292.0,0,Transient,58.00,1,1
1,HB-018087,Resort Hotel,0,6,2023,November,44,1,2023-11-01,2,...,D,3,No Deposit,240.0,NaN,0,Transient,58.00,0,2
2,HB-022870,Resort Hotel,0,45,2024,April,15,8,2024-04-08,0,...,D,1,No Deposit,240.0,NaN,0,Transient-Party,65.00,0,2
3,HB-048154,City Hotel,0,95,2024,March,11,17,2024-03-17,2,...,A,0,No Deposit,9.0,NaN,0,Transient,73.95,0,1
4,HB-060351,City Hotel,1,277,2024,November,45,7,2024-11-07,1,...,A,0,Non Refund,NaN,NaN,0,Transient,100.00,0,0


### Booking_id 

Se revisa la variable para evaluar si aporta informacion a la hora de realizar el analisis.

In [9]:
# Cantidad de registros en el dataset
registros = len(df)

# Cantidad de valores únicos en booking_id
booking_registros = df['booking_id'].nunique()

print(f"Registros totales en el Dataset: {registros} - Registro unicos en Booking_id: {booking_registros}")



Registros totales en el Dataset: 25000 - Registro unicos en Booking_id: 25000


La variable `booking_id` contiene un valor único para cada registro del dataset.
Al verificar la cantidad de valores únicos, se observa que coincide exactamente con la cantidad total de filas, lo que confirma que funciona únicamente como identificador.
Este tipo de variables no guarda relación con las demás características de la reserva y no aporta información útil para el análisis de cancelaciones. Solamente se trata de un identificador

Por lo tanto, `booking_id` no será utilizada en el análisis exploratorio y puede excluirse de las etapas de diagnóstico y limpieza.

In [10]:
#Se elimina booking_id
df.drop(columns=['booking_id'], inplace=True)

print("booking_id eliminado correctamente")


booking_id eliminado correctamente


Se registra el cambio en la bitácora

In [11]:
#Se registra el evento
registrar(
    problema="Variable sin aporte analítico",
    variable="booking_id",
    decision="Eliminación de la columna",
    justificacion="booking_id es un identificador único que no se relaciona con las demás variables y no aporta información para explicar cancelaciones."
)


---
### Hotel



In [12]:
# Diagnóstico de la variable hotel
print("Faltantes:")
print(df['hotel'].isnull().sum())

print("\nValores distintos")
print(df['hotel'].nunique())

print("\nValores de la variable:")
print(list(df['hotel'].unique()))

print("\nDistribución")
print(df['hotel'].value_counts())

Faltantes:
0

Valores distintos
2

Valores de la variable:
['Resort Hotel', 'City Hotel']

Distribución
hotel
City Hotel      16716
Resort Hotel     8284
Name: count, dtype: int64


**Diagnóstico de la variable `Hotel`**

**Completitud**  
La variable se encuentra completamente registrada en todas las filas del dataset. No presenta valores faltantes y no requiere ningún tipo de tratamiento adicional.

**Validez**  
Las dos categorías observadas *City Hotel* y *Resort Hotel* tienen con el contexto del conjunto de datos y no se detectan valores fuera de lo esperado ni categorías que indiquen errores de carga.

**Consistencia**  
Las categorías estan escritas de manera uniforme, sin variaciones en mayúsculas, espacios, guiones o errores de tipeo. 

La Variable `hotel` no requiere ningun tipo de tratamiento, por lo tanto no se asienta en la Bitacora.



---
### Is Cancelled 



In [13]:
print("Faltantes:")
print(df['is_canceled'].isnull().sum())

print("\nValores distintos:")
print(df['is_canceled'].nunique())

print("\nValores de la variable:")
print(list(df['is_canceled'].unique()))



Faltantes:
0

Valores distintos:
2

Valores de la variable:
[np.int64(0), np.int64(1)]


**Diagnóstico de la variable `is_canceled`**

**Completitud**  
La variable se encuentra completamente registrada en todas las filas del dataset. No presenta valores faltantes y no requiere tratamiento adicional en esta etapa.

**Validez**  
Los valores observados (0 y 1) tienen sentido en la variable. No se detectan valores fuera de este conjunto.

**Consistencia**  
La variable mantiene un formato uniforme en todo el dataset. No existen variantes como “Yes/No”, “Canceled/Not canceled” ni valores mal tipeados. Tampoco se observan inconsistencias entre registros que indiquen errores de carga.

**Unicidad**  
La variable no funciona como identificador ni presenta valores únicos por fila. Al contener únicamente dos categorías repetidas en miles de registros, su comportamiento es el esperado para una variable binaria y no cumple ninguna condición para ser considerada un ID.

### Conclusión  
La variable `is_canceled` se encuentra completa, válida, consistente y precisa. No requiere limpieza ni transformación en esta etapa.


---
### Lead Time



In [14]:
print("Faltantes:")
print(df['lead_time'].isnull().sum())

print("\nValores distintos:")
print(df['lead_time'].nunique())

print("\nTipos de Datos en la variable")
print(df['lead_time'].dtype)

print("\nValores negativos (Imposibles - Errores)")
print((df['lead_time'] < 0).sum())

print("\nValor mínimo:", df['lead_time'].min())
print("Valor máximo:", df['lead_time'].max())

#Calculo de Outliers con IQR

print("\nOutliers:")

Q1 = df['lead_time'].quantile(0.25)
Q3 = df['lead_time'].quantile(0.75)
IQR = Q3 - Q1

lim_sup = Q3 + 1.5 * IQR

print("\nLímite superior (IQR):", lim_sup)
print("Cantidad de outliers (IQR):", (df['lead_time'] > lim_sup).sum())

# Outliers con Z-score

mean = df['lead_time'].mean()
std = df['lead_time'].std()

lim_sup_z = mean + 3 * std

print("\nLímite superior (Z-score):", lim_sup_z)

outliers_z = df[df['lead_time'] > lim_sup_z]

print("Cantidad de outliers (Z-score):", len(outliers_z))




Faltantes:
0

Valores distintos:
460

Tipos de Datos en la variable
int64

Valores negativos (Imposibles - Errores)
0

Valor mínimo: 0
Valor máximo: 629

Outliers:

Límite superior (IQR): 370.5
Cantidad de outliers (IQR): 652

Límite superior (Z-score): 423.04081825080823
Cantidad de outliers (Z-score): 300


**Diagnóstico de la variable `lead_time`**

**Completitud**  
La variable no presenta valores faltantes. Todos los registros contienen un valor numérico válido que representa la cantidad de días entre la reserva y la fecha de llegada.

**Validez**  
El tipo de dato es `int64`, lo que nos dice que no tenemos ningun string con caracteres, son todos valores numericos. No se detectan valores negativos, lo cual confirma que no existen errores obvios de carga (el tiempo de anticipación no puede ser menor que cero).

**Rango de valores**  
El valor mínimo observado es **0**, correspondiente a reservas realizadas el mismo día de llegada.  
El valor máximo es **629**, lo que indica reservas hechas con más de un año y medio de anticipación. Este valor es posible dentro del contexto del negocio, aunque poco frecuente.

**Variabilidad**  
La variable presenta **460 valores distintos**, lo que indica una variabilidad amplia y útil para análisis estadístico. 

**Detección de outliers**

Se aplicaron dos métodos estadísticos para identificar valores atípicos: **IQR** y **Z‑score**.

**1. Método IQR (Interquartile Range)**  
- Q1 = 18  
- Q3 = 159  
- IQR = 141  
- **Límite superior:** 370.5  
- **Outliers detectados:** 652 registros

Todos los valores superiores a **370.5 días** se consideran outliers estadísticos.  
Estos valores no son errores, pero representan reservas realizadas con mucha anticipación y deben documentarse como casos extremos.

**2. Método Z‑score (Regla de los 3 sigmas)**  
- Media = 103.28  
- Desvío estándar = 106.58  
- **Límite superior:** 423.04  
- **Outliers detectados:** 300 registros

Los valores superiores a **423 días** se consideran outliers según este método.  
El método Z‑score detecta menos outliers porque su límite es más alto que el del IQR.

**Conclusión**

La variable `lead_time` es completa, válida y los datos tienen sentido.  
No presenta valores imposibles ni errores de carga.  
Ambos métodos estadísticos detectan valores extremadamente altos (entre 370 y 629 días), que deben considerarse como *outliers*.

Se agrega a la bitacora:

In [15]:
registrar(
    problema="Valores extremadamente altos (outliers)",
    variable="lead_time",
    decision="Revisar en etapa de transformación",
    justificacion="Valores por encima de 370–423 días son estadísticamente atípicos. No son errores, pero pueden distorsionar métricas."
)


---

### Arrival date year

Analisis de la variable

In [213]:
print("Faltantes:")
print(df['arrival_date_year'].isnull().sum())

print("\nValores distintos:")
print(df['arrival_date_year'].nunique())

print("\nValores de la variable:")
valores = sorted(df['arrival_date_year'].astype(int).unique().tolist())
print(valores)

print("\nTipos de Datos en la variable")
print(df['arrival_date_year'].dtype)

print("\nValores imposibles (años fuera de rango lógico)")
print(df[(df['arrival_date_year'] < 2000) | (df['arrival_date_year'] > 2026)].shape[0])

print("\nValor mínimo:", df['arrival_date_year'].min())
print("Valor máximo:", df['arrival_date_year'].max())



Faltantes:
0

Valores distintos:
3

Valores de la variable:
[2023, 2024, 2025]

Tipos de Datos en la variable
int64

Valores imposibles (años fuera de rango lógico)
0

Valor mínimo: 2023
Valor máximo: 2025



**Diagnóstico de la variable `arrival_date_year`**

**Completitud**  
La variable no presenta valores faltantes. Todos los registros contienen un año válido.

**Validez**  
El tipo de dato es `int64`, representan años del calendario. No se detectan valores imposibles: comprenden los años 2023 - 2025.

**Rango de valores**  
El valor mínimo es **2023** y el máximo es **2025**, coincidiendo con el período cubierto por el dataset. No se observan errores de carga ni valores erroneos.

**Conclusión**  
La variable `arrival_date_year` es completa, válida y consistente. No presenta valores imposibles, errores de carga ni outliers. No requiere acciones de limpieza.


---

### Arrival date month

Analisis de la variable

In [214]:
print("Faltantes:")
print(df['arrival_date_month'].isnull().sum())

print("\nValores distintos:")
print(df['arrival_date_month'].nunique())

print("\nValores de la variable:")
print(sorted(df['arrival_date_month'].unique()))

print("\nTipos de Datos en la variable")
print(df['arrival_date_month'].dtype)

#Lista con los nombres correctos de los meses
print("\nValores imposibles (meses mal cargados)")
meses_validos = [
    "January","February","March","April","May","June",
    "July","August","September","October","November","December"
]

#Busca valores que NO esten en la lista de los meses correctos
print(df[~df['arrival_date_month'].isin(meses_validos)].shape[0])


Faltantes:
0

Valores distintos:
12

Valores de la variable:
['April', 'August', 'December', 'February', 'January', 'July', 'June', 'March', 'May', 'November', 'October', 'September']

Tipos de Datos en la variable
str

Valores imposibles (meses mal cargados)
0


**Diagnóstico de la variable `arrival_date_month`**

**Completitud**  
La variable no presenta valores faltantes. Todos los registros contienen un mes válido.

**Validez**  
El tipo de dato es `str`, representan los meses como texto. No se detectan valores erroneos: los 12 meses del año están correctamente escritos y sin abreviaturas ni errores tipográficos.

**Variabilidad**  
La variable presenta **12 valores distintos**, correspondientes a los meses del año. 

**Valores de la variable**  
Los meses presentes son: April, August, December, February, January, July, June, March, May, November, October y September. Todos son meses válidos.

**Conclusión**  
La variable `arrival_date_month` es completa, válida y consistente. No presenta valores imposibles, errores de carga ni problemas de formato. No requiere acciones de limpieza.


---

### arrival_date_week_number

Analisis de la variable


In [ ]:
print("Faltantes:")
print(df['arrival_date_week_number'].isnull().sum())

print("\nValores distintos:")
print(df['arrival_date_week_number'].nunique())

print("\nValores de la variable:")

valores = sorted(df['arrival_date_week_number'].astype(int).unique().tolist())
print(valores)

print("\nTipos de Datos en la variable")
print(df['arrival_date_week_number'].dtype)

print("\nValores imposibles (semanas fuera de rango 1-52)")
print(df[(df['arrival_date_week_number'] < 1) | (df['arrival_date_week_number'] > 52)].shape[0])

print("\nValor mínimo:", df['arrival_date_week_number'].min())
print("Valor máximo:", df['arrival_date_week_number'].max())


Faltantes:
0

Valores distintos:
52

Valores de la variable:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52]

Tipos de Datos en la variable
int64

Valores imposibles (semanas fuera de rango 1-52)
0

Valor mínimo: 1
Valor máximo: 52



**Diagnóstico de la variable `arrival_date_week_number`**

**Completitud**  
La variable no presenta valores faltantes. Todos los registros contienen un número de semana válido.

**Validez**  
El tipo de dato es `int64`, representando correctamente semanas del año como números enteros. No se detectan valores erróneos, todas las semanas están dentro del rango lógico (1 a 52).

**Variabilidad**  
La variable presenta *52 valores distintos*, correspondientes a todas las semanas del año. Esto confirma que la codificación es completa y consistente.

**Valores de la variable**  
Los valores presentes abarcan todas las semanas del año, desde la *1* hasta la *52*, sin semanas faltantes ni errores.

**Conclusión**  
La variable `arrival_date_week_number` es completa, válida y consistente. No presenta valores imposibles, errores de carga ni problemas de formato.


Se evalua si la variable aporta informacion a la hora de realizar el analisis.

La variable `arrival_date_week_number` indica el número de semana del año, pero su información es redundante comparado a las variables `arrival_date_year`, `arrival_date_month` y `arrival_date_day_of_month`, que ya permiten analizar el comportamiento temporal de las reservas. Es una variable para uso interno de la empresa, que no aporta información adicional relevante para el estudio de cancelaciones, se debe decidir si amerita eliminacion.

In [228]:
registrar(
    problema="Variable redundante",
    variable="arrival_date_week_number",
    decision="Revisar en etapa de transformación",
    justificacion="La información de semana del año es redundante frente a year, month y day_of_month. Se evaluará su eliminación en la próxima etapa."
)



---

### Arrival date day of month

Analisis de la variable

In [240]:
print("Faltantes:")
print(df['arrival_date_day_of_month'].isnull().sum())

print("\nValores distintos:")
print(df['arrival_date_day_of_month'].nunique())

print("\nValores de la variable:")
valores = sorted(df['arrival_date_day_of_month'].astype(int).unique().tolist())
print(valores)

print("\nTipos de Datos en la variable")
print(df['arrival_date_day_of_month'].dtype)

print("\nValores imposibles (días fuera de rango 1-31)")
print(df[(df['arrival_date_day_of_month'] < 1) | (df['arrival_date_day_of_month'] > 31)].shape[0])

print("\nValor mínimo:", df['arrival_date_day_of_month'].min())
print("Valor máximo:", df['arrival_date_day_of_month'].max())


Faltantes:
0

Valores distintos:
31

Valores de la variable:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]

Tipos de Datos en la variable
int64

Valores imposibles (días fuera de rango 1-31)
0

Valor mínimo: 1
Valor máximo: 31


**Diagnóstico de la variable `arrival_date_day_of_month`**

**Completitud**  
La variable no presenta valores faltantes. Todos los registros contienen un día válido del mes.

**Validez**  
El tipo de dato es `int64`, representando correctamente días del mes como números enteros. No se detectan valores fuera del rango lógico (1 a 31).

**Variabilidad**  
La variable presenta *31 valores distintos*, correspondientes a todos los días posibles del mes.

**Valores de la variable**  
Los valores presentes van desde el día *1* hasta el *31*, sin saltos ni valores imposibles.

**Evaluación de aporte analítico**  
La variable `arrival_date_day_of_month` aporta información temporal detallada que puede ser útil para analizar patrones de reservas y cancelaciones dentro de cada mes. No es redundante frente a otras variables temporales, ya que permite estudiar comportamientos específicos según el día del mes.

**Conclusión**  
La variable `arrival_date_day_of_month` es completa, válida y consistente. No presenta valores imposibles ni errores de carga. Se considera útil para el análisis.


---

### Arrival Date

Se revisa la variable verficiar su integridad y evaluar si aporta informacion a la hora de realizar el analisis.

In [338]:
print("Faltantes:")
print(df['arrival_date'].isnull().sum())

print("\nValores distintos:")
print(df['arrival_date'].nunique())

print("\nTipos de Datos en la variable")
print(df['arrival_date'].dtype)


#Valores erroneos

#Patron REGEX para verificar que las fechas sean todas correctas
patron = r'^\d{4}-\d{2}-\d{2}$'

invalidos = df[~df['arrival_date'].astype(str).str.match(patron)]

print("\nFechas con formato inválido (aaaa-mm-dd):", invalidos.shape[0])
invalidos.head()

print("\nValores imposibles (fechas fuera de rango lógico)")
# Se utiliza un rango que sabemos que el dataset tiene que manejar
print(df[(df['arrival_date'] < '2020-01-01') | (df['arrival_date'] > '2026-12-31')].shape[0])

print("\nValor mínimo:", df['arrival_date'].min())
print("Valor máximo:", df['arrival_date'].max())


Faltantes:
0

Valores distintos:
793

Tipos de Datos en la variable
str

Fechas con formato inválido (aaaa-mm-dd): 0

Valores imposibles (fechas fuera de rango lógico)
0

Valor mínimo: 2023-07-01
Valor máximo: 2025-08-31


**Conclusion del analisis**

La variable `arrival_date` es válida y consistente, pero no aporta información adicional relevante para el análisis de cancelaciones.  

In [339]:
df[['arrival_date', 
    'arrival_date_year', 
    'arrival_date_month', 
    'arrival_date_day_of_month',
    'arrival_date_week_number'
]].head()


,arrival_date,arrival_date_year,arrival_date_month,arrival_date_day_of_month,arrival_date_week_number
0,2025-02-21,2025,February,21,8
1,2023-11-01,2023,November,1,44
2,2024-04-08,2024,April,8,15
3,2024-03-17,2024,March,17,11
4,2024-11-07,2024,November,7,45


La columna `arrival_date` contiene la fecha completa de llegada, pero su información ya está desagregada en `arrival_date_year`, `arrival_date_month`, `arrival_date_day_of_month`

Para el analisis, `arrival_date` es redundante, ya que no aporta datos nuevos respecto a las demás variables con datos de fechas que se pueden utilizar para analizar las temporadas del año, mediado del mes de la reserva, años con más o menos reservas o cancelaciones. 

Por este motivo, se propone su eliminación en la etapa de transformación.


Se registra el cambio en la bitácora

In [350]:
registrar(
    problema="Variable redundante",
    variable="arrival_date",
    decision="Revisar en etapa de transformación",
    justificacion="La fecha completa no aporta información adicional frente a year, month y day_of_month. Se evaluará su eliminación en la próxima etapa."
)


### stays in weekend nights

Analisis de la variable

In [374]:
print("Faltantes:")
print(df['stays_in_weekend_nights'].isnull().sum())

print("\nValores distintos:")
print(df['stays_in_weekend_nights'].nunique())

print("\nValores de la variable:")
valores = sorted(df['stays_in_weekend_nights'].astype(int).unique().tolist())
print(valores)

print("\nTipos de Datos en la variable")
print(df['stays_in_weekend_nights'].dtype)

print("\nValores imposibles (noches negativas)")
print((df['stays_in_weekend_nights'] < 0).sum())

print("\nValor mínimo:", df['stays_in_weekend_nights'].min())
print("Valor máximo:", df['stays_in_weekend_nights'].max())

#Outliers

#Calculo de Outliers con IQR

print("\nOutliers:")

Q1 = df['stays_in_weekend_nights'].quantile(0.25)
Q3 = df['stays_in_weekend_nights'].quantile(0.75)
IQR = Q3 - Q1

lim_sup = Q3 + 1.5 * IQR

print("\nLímite superior (IQR):", lim_sup)
print("Cantidad de outliers (IQR):", (df['stays_in_weekend_nights'] > lim_sup).sum())

# Outliers con Z-score

mean = df['stays_in_weekend_nights'].mean()
std = df['stays_in_weekend_nights'].std()

lim_sup_z = mean + 3 * std

print("\nLímite superior (Z-score):", lim_sup_z)

outliers_z = df[df['stays_in_weekend_nights'] > lim_sup_z]

print("Cantidad de outliers (Z-score):", len(outliers_z))


Faltantes:
0

Valores distintos:
12

Valores de la variable:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 14]

Tipos de Datos en la variable
int64

Valores imposibles (noches negativas)
0

Valor mínimo: 0
Valor máximo: 14

Outliers:

Límite superior (IQR): 5.0
Cantidad de outliers (IQR): 59

Límite superior (Z-score): 3.890320161777353
Cantidad de outliers (Z-score): 466


**Diagnóstico de la variable `stays_in_weekend_nights`**

**Completitud**  
La variable no presenta valores faltantes. Todos los registros contienen un número válido de noches de fin de semana.

**Validez**  
El tipo de dato es `int64`, representando correctamente cantidades enteras de noches. No se detectan valores imposibles: no hay noches negativas.


**Detección de outliers**  
Se evaluaron posibles valores atípicos mediante dos métodos:

- **IQR (Interquartile Range)**  
  - Límite superior: **5.0**  
  - Cantidad de outliers: **59**  
  Estos valores corresponden a huéspedes que se alojaron más de 5 noches en fin de semana. Son casos poco frecuentes, pero posibles.

- **Z-score (3 desviaciones estándar)**  
  - Límite superior: **3.89**  
  - Cantidad de outliers: **466**  
  Este método considera como atípicos valores moderados (4, 5, 6 noches) debido a la fuerte concentración en 0, 1 y 2 noches.  
  Sin embargo, estos valores siguen siendo posibles y no representan errores de carga.

En ambos métodos, los valores detectados como outliers validos y reflejan estancias inusualmente largas, pero posibles.

**Conclusión**  

`stays_in_weekend_nights` esta completa, válida y útil para el análisis.  
Los valores considerados outliers son plausibles y no requieren eliminación.  
La variable  aporta información relevante para el análisis del comportamiento de los huéspedes.  
Permite estudiar patrones de demanda en fines de semana, diferencias entre reservas cortas y largas, y posibles relaciones con cancelaciones o estacionalidad.  







---

### Stays in week nights

Analisis de la variable

In [382]:
print("Faltantes:")
print(df['stays_in_week_nights'].isnull().sum())

print("\nValores distintos:")
print(df['stays_in_week_nights'].nunique())

print("\nValores de la variable:")
valores = sorted(df['stays_in_week_nights'].astype(int).unique().tolist())
print(valores)

print("\nTipos de Datos en la variable")
print(df['stays_in_week_nights'].dtype)

print("\nValores imposibles (noches negativas)")
print((df['stays_in_week_nights'] < 0).sum())

print("\nValor mínimo:", df['stays_in_week_nights'].min())
print("Valor máximo:", df['stays_in_week_nights'].max())

# Outliers con IQR
print("\nOutliers (IQR):")

Q1 = df['stays_in_week_nights'].quantile(0.25)
Q3 = df['stays_in_week_nights'].quantile(0.75)
IQR = Q3 - Q1

lim_sup = Q3 + 1.5 * IQR

print("Límite superior (IQR):", lim_sup)
print("Cantidad de outliers (IQR):", (df['stays_in_week_nights'] > lim_sup).sum())

# Outliers con Z-score
mean = df['stays_in_week_nights'].mean()
std = df['stays_in_week_nights'].std()

lim_sup_z = mean + 3 * std

print("\nLímite superior (Z-score):", lim_sup_z)
print("Cantidad de outliers (Z-score):", (df['stays_in_week_nights'] > lim_sup_z).sum())


Faltantes:
0

Valores distintos:
25

Valores de la variable:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 34]

Tipos de Datos en la variable
int64

Valores imposibles (noches negativas)
0

Valor mínimo: 0
Valor máximo: 34

Outliers (IQR):
Límite superior (IQR): 6.0
Cantidad de outliers (IQR): 662

Límite superior (Z-score): 8.115568076228653
Cantidad de outliers (Z-score): 349


**Diagnóstico de la variable `stays_in_week_nights`**

**Completitud**  
La variable no presenta valores faltantes. Todos los registros contienen un número válido de noches entre semana.

**Validez**  
El tipo de dato es `int64`, representando correctamente cantidades enteras de noches. No se detectan valores imposibles, no hay noches negativas.

**Variabilidad**  
La variable presenta **25 valores distintos**, lo cual es esperable dado que representa la cantidad de noches entre semana que un huésped permanece en el hotel.

**Valores de la variable**  
Los valores van desde **0** hasta **34** noches.  
La distribución muestra que la mayoría de los huéspedes se alojan entre 1 y 3 noches entre semana, aunque existen estancias prolongadas que llegan a más de 20 noches.

**Detección de outliers**  
Se evaluaron posibles valores atípicos mediante dos métodos:

- **IQR (Interquartile Range)**  
  - Límite superior: *6.0*  
  - Cantidad de outliers: *662*  
  Estos valores corresponden a huéspedes con estancias largas entre semana. Son poco frecuentes, pero plausibles.

- **Z-score (3 desviaciones estándar)**  
  - Límite superior: *8.11*  
  - Cantidad de outliers: *349*  
  Este método marca como atípicos valores altos debido a la fuerte concentración en estancias cortas.  
  Sin embargo, los valores detectados siguen siendo posibles en reservas extendidas.

En ambos métodos, los valores considerados outliers **son válidos** y no representan errores de carga.

**Conclusión**  
`stays_in_week_nights` es completa, válida y útil para el análisis.  
Los valores considerados outliers son plausibles y no requieren eliminación.  
La variable aporta información esencial para el análisis del comportamiento de los huéspedes.  
Permite estudiar patrones de demanda entre semana, identificar estancias prolongadas, analizar diferencias entre reservas cortas y largas, y evaluar su relación con cancelaciones y estacionalidad.  






---

### Adults

Analisis de la variable

In [ ]:
print("Faltantes:")
print(df['adults'].isnull().sum())

print("\nValores distintos:")
print(df['adults'].nunique())

print("\nValores de la variable:")
valores = sorted(df['adults'].astype(int).unique().tolist())
print(valores)

print("\nTipos de Datos en la variable")
print(df['adults'].dtype)

print("\nValores imposibles (negativos)")
print((df['adults'] < 0).sum())

print("\nValor mínimo:", df['adults'].min())
print("Valor máximo:", df['adults'].max())

# Outliers con IQR
print("\nOutliers (IQR):")

Q1 = df['adults'].quantile(0.25)
Q3 = df['adults'].quantile(0.75)
IQR = Q3 - Q1

lim_sup = Q3 + 1.5 * IQR

print("Límite superior (IQR):", lim_sup)
print("Cantidad de outliers (IQR):", (df['adults'] > lim_sup).sum())

# Outliers con Z-score
mean = df['adults'].mean()
std = df['adults'].std()

lim_sup_z = mean + 3 * std

print("\nLímite superior (Z-score):", lim_sup_z)
print("Cantidad de outliers (Z-score):", (df['adults'] > lim_sup_z).sum())


Faltantes:
0

Valores distintos:
7

Valores de la variable:
[0, 1, 2, 3, 4, 5, 50]

Tipos de Datos en la variable
int64

Valores imposibles (adultos negativos)
0

Valor mínimo: 0
Valor máximo: 50

Outliers (IQR):
Límite superior (IQR): 2.0
Cantidad de outliers (IQR): 1318

Límite superior (Z-score): 3.5857797432390566
Cantidad de outliers (Z-score): 16


**Diagnóstico de la variable `adults`**

**Completitud**  
La variable no presenta valores faltantes. Todos los registros contienen un número válido (Integer).

**Validez**  
El tipo de dato es `int64`, representando correctamente cantidades enteras de personas adultas.  
No se detectan valores imposibles: no hay valores negativos.

**Variabilidad**  
La variable presenta **7 valores distintos**, lo cual es esperable para una variable que representa la cantidad de adultos por reserva.

**Valores de la variable**  
Los valores van desde **0** hasta **50** adultos.  
La distribución muestra que la mayoría de las reservas incluyen entre 1 y 2 adultos, mientras que los valores altos corresponden a reservas grupales o eventos.

**Detección de outliers**  
Se evaluaron posibles valores atípicos mediante dos métodos:

- **IQR (Interquartile Range)**  
  - Límite superior: **2.0**  
  - Cantidad de outliers: **1318**  
  Estos valores corresponden a reservas con más de 2 adultos. Son poco frecuentes, pero plausibles en reservas grupales o eventos.

- **Z-score (3 desviaciones estándar)**  
  - Límite superior: **3.58**  
  - Cantidad de outliers: **16**  
  Este método identifica únicamente los valores extremadamente altos (como 50 adultos).  
  Aunque son casos raros, siguen siendo posibles en contextos de grupos grandes.

En ambos métodos, los valores considerados outliers **son válidos** pero tiene que tenerse en cuenta al momento de sacar estadisiticas el outlier extremo de 50.

**Conclusión**  
La variable `adults` es completa, válida y útil para el análisis.  
Los valores considerados outliers son plausibles (Aunque el valor de 50 debe considerarse durante el analisis).  
La variable aporta información esencial para el estudio de la composición de las reservas, permitiendo analizar patrones de ocupación, diferencias entre reservas individuales y grupales, y su relación con cancelaciones y estacionalidad.  

Se registra el outlier extremo en la Bitacora


In [398]:
registrar(
    problema="Outlier extremo en variable adults",
    variable="adults",
    decision="Revisar en etapa de transformación",
    justificacion=(
        "Se detecta un valor de 50 adultos en una sola reserva, este valor es plausible pero extremadamente raro. Se recomienda revisar su impacto en el análisis y considerar tratamiento especial si distorsiona las metricas."
    )
)


--- 

### Children

Analisis de la variable 

In [414]:
print("Faltantes:")
print(df['children'].isnull().sum())

print("\nValores distintos:")
print(df['children'].nunique())

print("\nValores de la variable:")
valores = sorted(df['children'].unique().tolist())
print(valores)

print("\nTipos de Datos en la variable")
print(df['children'].dtype)

print("\nValores imposibles (niños negativos)")
print((df['children'] < 0).sum())

print("\nValor mínimo:", df['children'].min())
print("Valor máximo:", df['children'].max())

# Outliers con IQR
print("\nOutliers (IQR):")

Q1 = df['children'].quantile(0.25)
Q3 = df['children'].quantile(0.75)
IQR = Q3 - Q1

lim_sup = Q3 + 1.5 * IQR

print("Límite superior (IQR):", lim_sup)
print("Cantidad de outliers (IQR):", (df['children'] > lim_sup).sum())

print("Faltantes:")
print(df['children'].isnull().sum())

print("\nValores distintos:")
print(df['children'].nunique())

print("\nValores de la variable:")
valores = sorted(df['children'].unique().tolist())
print(valores)

print("\nTipos de Datos en la variable")
print(df['children'].dtype)

print("\nValores imposibles (negativos)")
print((df['children'] < 0).sum())

print("\nValor mínimo:", df['children'].min())
print("Valor máximo:", df['children'].max())

# Outliers con IQR
print("\nOutliers (IQR):")

Q1 = df['children'].quantile(0.25)
Q3 = df['children'].quantile(0.75)
IQR = Q3 - Q1

lim_sup = Q3 + 1.5 * IQR

print("Límite superior (IQR):", lim_sup)
print("Cantidad de outliers (IQR):", (df['children'] > lim_sup).sum())

# Outliers con Z-score
mean = df['children'].mean()
std = df['children'].std()

lim_sup_z = mean + 3 * std

print("\nLímite superior (Z-score):", lim_sup_z)
print("Cantidad de outliers (Z-score):", (df['children'] > lim_sup_z).sum())

Faltantes:
0

Valores distintos:
4

Valores de la variable:
[0.0, 1.0, 2.0, 3.0]

Tipos de Datos en la variable
float64

Valores imposibles (niños negativos)
0

Valor mínimo: 0.0
Valor máximo: 3.0

Outliers (IQR):
Límite superior (IQR): 0.0
Cantidad de outliers (IQR): 1773
Faltantes:
0

Valores distintos:
4

Valores de la variable:
[0.0, 1.0, 2.0, 3.0]

Tipos de Datos en la variable
float64

Valores imposibles (negativos)
0

Valor mínimo: 0.0
Valor máximo: 3.0

Outliers (IQR):
Límite superior (IQR): 0.0
Cantidad de outliers (IQR): 1773

Límite superior (Z-score): 1.2742140998654448
Cantidad de outliers (Z-score): 740


**Diagnóstico de la variable `children`**

**Completitud**  
La variable no presenta valores faltantes. Todos los registros contienen un número válido de niños asociados a la reserva.

**Validez**  
El tipo de dato es `float64`, no se detectan valores imposibles, no hay valores negativos.

**Variabilidad**  
La variable presenta **4 valores distintos**: 0, 1, 2 y 3 niños por reserva.  
La distribución está fuertemente concentrada en **0 niños**, lo cual es esperable en reservas hoteleras.

**Valores de la variable**  
Los valores van desde **0** hasta **3** niños.  
La mayoría de las reservas no incluyen menores, mientras que los valores de 1 y 2 niños representan casos familiares.  
El valor de 3 niños es extremadamente raro.

**Detección de outliers**  
Se evaluaron posibles valores atípicos mediante dos métodos:

- **IQR (Interquartile Range)**  
  - Límite superior: **0.0**  
  - Cantidad de outliers: **1773**  
  Debido a que la mayoría de los registros tienen 0 niños, cualquier valor mayor a 0 queda marcado como outlier por este método.  
  No es un error, refleja la fuerte concentración en el valor 0.

- **Z-score (3 desviaciones estándar)**  
  - Límite superior: **1.27**  
  - Cantidad de outliers: **740**  
  Este método identifica como atípicos los valores más altos (2 y 3 niños).  
  Aunque son casos raros para el dataset, simplemente son familias numerosas.

En ambos métodos, los valores considerados outliers **son válidos** y no representan errores de carga.  

**Conclusión**  
La variable `children` es completa, válida y útil para el análisis.  
Los valores considerados outliers son plausibles y no requieren eliminación.  
La variable aporta información relevante para el estudio de la composición familiar de las reservas y su relación con cancelaciones, estacionalidad y tipo de estadía.  


--- 

### Babies

Analisis de la variable

In [422]:
print("Faltantes:")
print(df['babies'].isnull().sum())

print("\nValores distintos:")
print(df['babies'].nunique())

print("\nValores de la variable:")
valores = sorted(df['babies'].unique().tolist())
print(valores)

print("\nTipos de Datos en la variable")
print(df['babies'].dtype)

print("\nValores imposibles (negativos)")
print((df['babies'] < 0).sum())

print("\nValor mínimo:", df['babies'].min())
print("Valor máximo:", df['babies'].max())

# Outliers con IQR
print("\nOutliers (IQR):")

Q1 = df['babies'].quantile(0.25)
Q3 = df['babies'].quantile(0.75)
IQR = Q3 - Q1

lim_sup = Q3 + 1.5 * IQR

print("Límite superior (IQR):", lim_sup)
print("Cantidad de outliers (IQR):", (df['babies'] > lim_sup).sum())

# Outliers con Z-score
mean = df['babies'].mean()
std = df['babies'].std()

lim_sup_z = mean + 3 * std

print("\nLímite superior (Z-score):", lim_sup_z)
print("Cantidad de outliers (Z-score):", (df['babies'] > lim_sup_z).sum())


Faltantes:
0

Valores distintos:
5

Valores de la variable:
[0, 1, 2, 9, 10]

Tipos de Datos en la variable
int64

Valores imposibles (negativos)
0

Valor mínimo: 0
Valor máximo: 10

Outliers (IQR):
Límite superior (IQR): 0.0
Cantidad de outliers (IQR): 170

Límite superior (Z-score): 0.3643382353624856
Cantidad de outliers (Z-score): 170


**Diagnóstico de la variable `babies`**

**Completitud**  
La variable no presenta valores faltantes. Todos los registros contienen un número válido de bebés asociados a la reserva.

**Validez**  
El tipo de dato es `int64`, representando correctamente cantidades enteras de bebés.  
No se detectan valores imposibles, no hay valores negativos.

**Variabilidad**  
La variable presenta **5 valores distintos**: 0, 1, 2, 9 y 10 bebés por reserva.  
La distribución está fuertemente concentrada en **0 bebés**, lo cual es esperable en reservas hoteleras.  
Los valores 9 y 10 son extremadamente raros y corresponden a casos muy particulares.

**Valores de la variable**  
Los valores van desde **0** hasta **10** bebés.  
La mayoría de las reservas no incluyen menores de un año.  
Los valores 1 y 2 representan casos familiares típicos.  
Los valores 9 y 10 son inusuales y probablemente correspondan a reservas grupales o errores de carga, aunque no pueden descartarse sin más contexto.

**Detección de outliers**  
Se evaluaron posibles valores atípicos mediante dos métodos:

- **IQR (Interquartile Range)**  
  - Límite superior: **0.0**  
  - Cantidad de outliers: **170**  
  Debido a que la mayoría de los registros tienen 0 bebés, cualquier valor mayor a 0 queda marcado como outlier por este método.  
  Esto no es un error, la concentración esta en el valor 0.

- **Z-score (3 desviaciones estándar)**  
  - Límite superior: **0.3643**  
  - Cantidad de outliers: **170**  
  Este método también identifica como atípicos todos los valores mayores a 0.  
  Los valores altos (9 y 10) son extremadamente raros y podrían requerir verificación adicional, aunque siguen siendo posibles en reservas grupales.

En ambos métodos, los valores considerados outliers **son válidos** y no representan necesariamente errores de carga aunque existen outliers extremos, son valores altos, pero no imposibles.

**Conclusión**  
La variable `babies` es completa, válida y útil para el análisis.  
Los valores considerados outliers son plausibles, pero pueden sesgar las estadistias.  
La variable aporta información relevante para el estudio de la composición familiar de las reservas y permite detectar casos especiales como reservas grupales o estancias familiares extensas.  


In [430]:
registrar(
    problema="Valores atípicos extremos",
    variable="babies",
    decision="Revisar en etapa de transformación",
    justificacion="La variable presenta valores válidos pero inusuales (9 y 10 bebés). Aunque no son imposibles, su frecuencia es extremadamente baja y podrían corresponder a reservas grupales o errores de carga. Se revisarán en la próxima etapa."
)


--- 

### Meal

Analisis de la variable 

In [ ]:

print("Faltantes:")
print(df['meal'].isnull().sum())

print("\nValores distintos")
print(df['meal'].nunique())

print("\nValores de la variable:")
valores = sorted(df['meal'].unique().tolist())
print(valores)

print("\nVariables que son undefined: ")
lista_comidas_correctas = ['BB', 'FB', 'HB', 'SC']
#Busca valores que NO esten en la lista de comidas correctas
print(df[~df['meal'].isin(lista_comidas_correctas)].shape[0])




Faltantes:
0

Valores distintos
5

Valores de la variable:
['BB', 'FB', 'HB', 'SC', 'Undefined']

Variables que son undefined: 
228


**Diagnóstico de la variable `meal`**

**Completitud**  
La variable no presenta valores faltantes. El 100% de las reservas tiene un régimen alimenticio registrado.

**Validez**  
El tipo de dato es categórico y todos los valores corresponden a códigos válidos del dominio hotelero: `BB`, `FB`, `HB`, `SC`.  
Se detecta además la categoría **`Undefined`**, que representa valores no especificados por el sistema.

**Variabilidad**  
La variable presenta **5 valores distintos**: `BB`, `FB`, `HB`, `SC` y `Undefined`.  
La distribución está fuertemente concentrada en **BB (Bed & Breakfast)**, que es el régimen más común en hoteles.

**Valores de la variable**  
Los valores encontrados son:  
`['BB', 'FB', 'HB', 'SC', 'Undefined']`  
La categoría **Undefined** aparece en **228 registros**, lo cual representa un pequeño porcentaje del dataset, pero suficiente para requerir revisión.

**Detección de valores inválidos o especiales**  
Aunque no existen valores imposibles, la presencia de **Undefined** indica un caso especial:  
- Puede representar datos faltantes codificados.  
- Puede ser una categoría residual del sistema de origen.  
- Puede requerir recodificación en la etapa de transformación.

No se detectan outliers numéricos porque la variable es categórica.

**Conclusión**  
La variable `meal` es completa y válida en términos generales.  
La categoría **Undefined** debe ser revisada en la etapa de transformación para decidir si se recodifica como valor faltante, se agrupa dentro de “Other”, o se elimina.


Se agrega a la bitacora:

In [494]:
registrar(
    problema="Categoría especial 'Undefined'",
    variable="meal",
    decision="Revisar en etapa de transformación",
    justificacion="La variable contiene 228 registros con el valor 'Undefined', que no corresponde a un régimen alimenticio estándar. Se evaluará si debe recodificarse como NA, agruparse en 'Other' o eliminarse."
)


---

### Country

Analisis de la variable

In [544]:

print("Faltantes:")
print(df['country'].isnull().sum())

print("\nValores distintos")
print(df['country'].nunique())

print("\nTipos de Datos en la variable")
print(df['country'].dtype)

print("\nValores de la variable:")
valores = sorted(df['country'].dropna().unique().tolist())
print(valores)



Faltantes:
101

Valores distintos
129

Tipos de Datos en la variable
str

Valores de la variable:
['ABW', 'AGO', 'ALB', 'AND', 'ARE', 'ARG', 'ARM', 'ATA', 'ATF', 'AUS', 'AUT', 'AZE', 'BEL', 'BEN', 'BGD', 'BGR', 'BIH', 'BLR', 'BOL', 'BRA', 'CHE', 'CHL', 'CHN', 'CMR', 'CN', 'COL', 'CPV', 'CRI', 'CUB', 'CYP', 'CZE', 'DEU', 'DMA', 'DNK', 'DOM', 'DZA', 'ECU', 'EGY', 'ESP', 'EST', 'FIN', 'FRA', 'FRO', 'GBR', 'GEO', 'GHA', 'GIB', 'GLP', 'GNB', 'GRC', 'GTM', 'HKG', 'HRV', 'HUN', 'IDN', 'IMN', 'IND', 'IRL', 'IRN', 'IRQ', 'ISL', 'ISR', 'ITA', 'JAM', 'JEY', 'JOR', 'JPN', 'KAZ', 'KOR', 'KWT', 'LAO', 'LBN', 'LBY', 'LCA', 'LIE', 'LKA', 'LTU', 'LUX', 'LVA', 'MAC', 'MAR', 'MCO', 'MEX', 'MKD', 'MLT', 'MOZ', 'MUS', 'MYS', 'MYT', 'NGA', 'NLD', 'NOR', 'NZL', 'OMN', 'PAK', 'PAN', 'PER', 'PHL', 'POL', 'PRI', 'PRT', 'PRY', 'QAT', 'ROU', 'RUS', 'SAU', 'SEN', 'SGP', 'SLE', 'SRB', 'SVK', 'SVN', 'SWE', 'SYC', 'TGO', 'THA', 'TMP', 'TUN', 'TUR', 'TWN', 'TZA', 'UKR', 'URY', 'USA', 'VEN', 'VGB', 'VNM', 'ZAF', 'ZWE']

**Diagnóstico de la variable `country`**

**Completitud**  
La variable presenta **101 valores faltantes**, lo que representa menos del **1%** del dataset.  
Aunque el porcentaje es bajo, los NA deben revisarse porque `country` es una variable relevante para segmentación y análisis de comportamiento de huéspedes.

**Validez**  
El tipo de dato es categórico (`str`).  
Los valores corresponden a **códigos ISO de países**.  
No se detectan valores imposibles ni códigos mal formados.

**Valores de la variable**  
La lista de países incluye códigos como:  
`['ABW', 'AGO', 'ALB', 'AND', 'ARE', 'ARG', 'ARM', ... 'ZAF', 'ZWE']`  
La presencia de muchos países con baja frecuencia es esperable y no representa un problema.

**Detección de outliers**  
La variable es categórica, por lo que **no aplica** la detección de outliers mediante métodos numéricos.  
Sin embargo, las categorías de muy baja frecuencia pueden considerarse **valores raros**, no errores.

**Conclusión**  
La variable `country` es válida y útil para análisis de segmentación, estacionalidad y comportamiento de cancelaciones.  
Los **101 valores faltantes** deben revisarse en la etapa de transformación para decidir si se imputan, se agrupan como “Unknown” o se mantienen como NA.  


Se agrega a la bitacora:

In [545]:
registrar(
    problema="Valores faltantes y alta diversidad",
    variable="country",
    decision="Revisar en etapa de transformación",
    justificacion="La variable presenta 101 valores faltantes y 129 países distintos. Los NA deben evaluarse para imputación o recodificación, y las categorías de baja frecuencia deben analizarse para decidir si se agrupan o se mantienen."
)


---

### Market segment

In [570]:
print("Faltantes:")
faltantes = df['market_segment'].isnull().sum()
print(f"{faltantes} ({faltantes/len(df)*100:.2f}%)")

print("\nValores distintos:")
distinct = df['market_segment'].nunique()
print(distinct)

print("\nValores de la variable:")
valores = sorted(df['market_segment'].unique().tolist())
print(valores)


Faltantes:
0 (0.00%)

Valores distintos:
7

Valores de la variable:
['Aviation', 'Complementary', 'Corporate', 'Direct', 'Groups', 'Offline TA/TO', 'Online TA']


**Diagnóstico de la variable `market_segment`**

**Completitud**  
La variable no presenta valores faltantes. El 100% de los registros tiene un segmento de mercado asignado.

**Validez**  
El tipo de dato es categórico (`str`).  
Las 7 categorías encontradas son:  
`['Aviation', 'Complementary', 'Corporate', 'Direct', 'Groups', 'Offline TA/TO', 'Online TA']`  
No se detectan valores inválidos.

**Posibles problemas detectados**  

- **Categorías de baja frecuencia**  
  - `Aviation` y `Complementary` suelen tener muy pocos registros.  
  - No son errores, pero pueden generar ruido en modelos predictivos o análisis estadísticos.

- **Segmentos ambiguos**  
  - `Complementary` puede incluir cortesías, estadías de personal o compensaciones.  
  - Su interpretación depende del sistema de origen y podría requerir recodificación.

- **Redundancia con otras variables**  
  - `market_segment` puede solaparse con `distribution_channel`.  
  - En la etapa de transformación se evaluará si ambas aportan información distinta o si conviene agruparlas.


**Conclusión**  
La variable `market_segment` es completa y válida.  
Las categorías minoritarias y las posibles ambigüedades deben revisarse en la etapa de transformación para decidir si se agrupan, se recodifican o se mantienen.

Se agrega a la bitacora:



In [578]:
registrar(
    problema="Categorías minoritarias y ambigüedad",
    variable="market_segment",
    decision="Revisar en etapa de transformación",
    justificacion="Existen categorías con muy baja frecuencia (Aviation, Complementary) y posibles solapamientos con distribution_channel. Se evaluará si conviene agruparlas o recodificarlas."
)


---

### Distribution Channel



---

**Agent**: Se revisa la variable para evaluar si aporta informacion a la hora de realizar el analisis.

In [579]:
print(f"Valores repetidos:{df['agent'].value_counts().head(10)}")

print(f"\nValores unicos: {df['agent'].nunique()}")


print(f"\nDatos Faltantes: {df['agent'].isnull().sum()}")


Valores repetidos:agent
9.0      6690
240.0    2896
1.0      1508
7.0       776
14.0      755
6.0       720
250.0     562
241.0     369
28.0      340
8.0       326
Name: count, dtype: int64

Valores unicos: 268

Datos Faltantes: 3489


La variable `agent` presenta *3489* valores faltantes, *268* valores únicos y miles de repeticiones de los mismos códigos, lo que muestra que se trata de un identificador interno del sistema de reservas. No representa una categoría interpretable ni aporta información relevante para explicar cancelaciones. Por lo tanto, se elimina del DataFrame por ser una variable administrativa sin valor analítico.

In [580]:
#Eliminacion de Agent
#df.drop(columns=['agent'], inplace=True)

#print("agent eliminado correctamente")
#TODO - no borrar, dejar para hacer una variable binaria mas adelante


Se registra el cambio en la bitácora

In [581]:
registrar(
    problema="Código interno sin valor analítico",
    variable="agent",
    decision="Eliminación de la columna",
    justificacion="Presenta muchos faltantes, 268 valores únicos y miles de repeticiones, lo que confirma que es un identificador interno sin aporte al análisis, simplemente una ID para el agente de la reserva."
)


**Company**: Se revisa la variable para evaluar si aporta informacion a la hora de realizar el analisis.

In [582]:
print(f"Valores repetidos:{df['company'].value_counts().head(10)}")

print(f"\nValores unicos: {df['company'].nunique()}")

print(f"\nDatos Faltantes: {df['company'].isnull().sum()}")


Valores repetidos:company
40.0     203
223.0    155
67.0      53
45.0      51
153.0     41
174.0     41
219.0     32
281.0     31
405.0     29
154.0     28
Name: count, dtype: int64

Valores unicos: 218

Datos Faltantes: 23574


La variable `company` presenta *23.574* valores faltantes y *218* valores únicos, además de múltiples repeticiones de los mismos códigos. Esto evidencia que se trata de un identificador interno del sistema de reservas, sin significado analítico ni relación con el comportamiento de cancelaciones. Por lo tanto, se elimina del DataFrame por ser una variable administrativa sin aporte al análisis.

In [583]:
#df.drop(columns=['company'], inplace=True)

#print("company eliminado correctamente")


Se registra el cambio en la bitácora

In [584]:
registrar(
    problema="Código interno sin valor analítico",
    variable="company",
    decision="Eliminación de la columna",
    justificacion="Presenta 23.574 faltantes, 218 valores únicos y repeticiones que confirman que es un identificador interno sin aporte al análisis."
)


### Variables eliminadas


- `booking_id`: identificador único sin aporte analítico.

- `arrival_date`: redundante frente a año, mes y día.

- `arrival_date_week_number`: no agrega información relevante para el analisis, es una variable interna para la administración.

- `agent`: código interno con 268 valores únicos y 3489 faltantes, sin significado analítico.

- `company`: código interno con 218 valores únicos y 23.574 faltantes, sin aporte al análisis.

In [585]:
print("Bitacora al momento, con las variables que se eliminaron:\n")
pd.DataFrame(registros_bitacora)


Bitacora al momento, con las variables que se eliminaron:



,problema_detectado,variable_afectada,decision_tomada,justificacion
0,Variable sin aporte analítico,booking_id,Eliminación de la columna,booking_id es un identificador único que no se...
1,Valores extremadamente altos (outliers),lead_time,Revisar en etapa de transformación,Valores por encima de 370–423 días son estadís...
2,Variable redundante,arrival_date_week_number,Eliminación de la columna,No aporta información adicional comparado a la...
3,Variable redundante,arrival_date,Eliminación de la columna,arrival_date no aporta información nueva en cu...
4,Código interno sin valor analítico,agent,Eliminación de la columna,"Presenta muchos faltantes, 268 valores únicos ..."
...,...,...,...,...
168,Código interno sin valor analítico,agent,Eliminación de la columna,"Presenta muchos faltantes, 268 valores únicos ..."
169,Código interno sin valor analítico,company,Eliminación de la columna,"Presenta 23.574 faltantes, 218 valores únicos ..."
170,Categorías minoritarias y ambigüedad,market_segment,Revisar en etapa de transformación,Existen categorías con muy baja frecuencia (Av...
171,Código interno sin valor analítico,agent,Eliminación de la columna,"Presenta muchos faltantes, 268 valores únicos ..."
